# Car Detection using HOG Features + Sliding Window

This notebook walks through a complete object detection pipeline for cars. The steps are:

1. Load and prepare the UIUC Car dataset
2. Extract HOG features from training patches
3. Train a linear SVM classifier
4. Detect cars in test images using a sliding window search
5. Evaluate detection performance

**Dataset:** UIUC Image Database for Car Detection  
Download: http://cogcomp.cs.illinois.edu/Data/Car/CarData.tar.gz

Run the following shell commands to get the data before executing any code cells:
```bash
wget http://cogcomp.cs.illinois.edu/Data/Car/CarData.tar.gz
tar -xzf CarData.tar.gz
```

In [ ]:
import os
import glob

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## 1. Data Preparation

Load the positive (car) and negative (non-car) training patches from `CarData/TrainImages/`. Each image is a grayscale PGM file with dimensions 100×40 pixels.

The function below reads all training images, assigns labels (1 = car, 0 = non-car), and returns the raw pixel arrays alongside the label vector.

In [ ]:
def load_training_data(train_dir='CarData/TrainImages'):
    """
    Load positive and negative training patches.

    Returns
    -------
    images : list of np.ndarray
    labels : list of int  (1 = car, 0 = non-car)
    """
    images = []
    labels = []

    # positive samples
    for path in sorted(glob.glob(os.path.join(train_dir, 'pos-*.pgm'))):
        pass

    # negative samples
    for path in sorted(glob.glob(os.path.join(train_dir, 'neg-*.pgm'))):
        pass

    return images, labels


def show_samples(images, labels, n=8):
    """Display a few positive and negative samples side by side."""
    pass


images, labels = load_training_data()
print(f'Loaded {len(images)} images ({labels.count(1) if hasattr(labels, "count") else sum(labels)} positive)')

## 2. HOG Feature Extraction

HOG (Histogram of Oriented Gradients) captures the distribution of gradient directions inside local cells of an image. It works well for rigid objects like cars because shape and edge information is preserved while ignoring color and lighting variation.

Key parameters chosen for this dataset:
- `orientations = 9` — number of gradient direction bins
- `pixels_per_cell = (8, 8)` — size of each local cell
- `cells_per_block = (2, 2)` — block normalisation extent

The function below extracts HOG features from a single grayscale patch and returns a flat feature vector.

In [ ]:
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    transform_sqrt=True,
)


def extract_hog_features(image):
    """
    Extract HOG feature vector from a grayscale image patch.

    Parameters
    ----------
    image : np.ndarray, shape (H, W)

    Returns
    -------
    features : np.ndarray, 1-D
    """
    pass


def build_feature_matrix(images):
    """
    Run extract_hog_features over a list of images and stack into a 2-D array.

    Returns
    -------
    X : np.ndarray, shape (n_samples, n_features)
    """
    pass


X = build_feature_matrix(images)
y = np.array(labels)
print(f'Feature matrix shape: {X.shape}')

## 3. SVM Training

A Linear SVM is a good fit for HOG features: the feature space is high-dimensional and the classes tend to be linearly separable after normalisation. `LinearSVC` from scikit-learn trains much faster than `SVC(kernel='linear')` for large feature vectors.

Steps:
1. Normalise features with `StandardScaler` (zero mean, unit variance).
2. Split into 80 % train / 20 % validation.
3. Fit `LinearSVC` and report validation accuracy.

In [ ]:
def train_classifier(X, y):
    """
    Scale features, split data, train LinearSVC, and return fitted objects.

    Returns
    -------
    clf     : fitted LinearSVC
    scaler  : fitted StandardScaler
    """
    scaler = StandardScaler()
    clf = LinearSVC(C=0.01, max_iter=5000)

    # TODO: fit scaler, split data, train clf, print validation report
    pass

    return clf, scaler


clf, scaler = train_classifier(X, y)

## 4. Sliding Window Detection

The sliding window approach works by:
1. Resizing the test image to several scales (image pyramid).
2. Sliding a fixed-size window (100×40, matching the training patch size) across the image with a chosen step size.
3. Extracting HOG features from each window and classifying it.
4. Collecting all windows where the classifier returns a positive prediction.
5. Applying Non-Maximum Suppression (NMS) to remove duplicate detections around the same car.

The three functions below implement these sub-steps.

In [ ]:
WINDOW_SIZE = (100, 40)   # (width, height) — matches training patch dimensions
STEP_SIZE   = (10, 10)    # pixels to move per step (x, y)
SCALES      = [1.0, 1.25, 1.5]


def sliding_window(image, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    """
    Generator that yields (x, y, window_patch) tuples.

    Parameters
    ----------
    image       : np.ndarray  (H, W) grayscale
    window_size : (w, h)
    step_size   : (dx, dy)
    """
    pass


def non_max_suppression(boxes, overlap_thresh=0.3):
    """
    Suppress overlapping bounding boxes, keeping only the most confident ones.

    Parameters
    ----------
    boxes         : np.ndarray, shape (N, 4) — [x1, y1, x2, y2]
    overlap_thresh : float, IoU threshold above which a box is suppressed

    Returns
    -------
    kept : np.ndarray, subset of boxes
    """
    pass


def detect_cars(image, clf, scaler, scales=SCALES):
    """
    Run multi-scale sliding window detection on a single test image.

    Returns
    -------
    detections : np.ndarray, shape (N, 4) — bounding boxes after NMS
    """
    detections = []

    for scale in scales:
        # resize image, slide window, classify patches, collect positives
        pass

    # apply NMS across all scales
    pass

    return detections

## 5. Evaluation

Run the detector on the test set (`CarData/TestImages/`) and measure performance. The UIUC dataset ships with ground-truth bounding boxes, so we can compute per-image precision and recall, then summarise across the full test set.

We'll also visualise a handful of test images with the predicted bounding boxes drawn on top to get a qualitative feel for where the detector is succeeding or failing.

In [ ]:
def load_test_images(test_dir='CarData/TestImages'):
    """
    Load test images and (optionally) ground-truth annotations.

    Returns
    -------
    test_images : list of (filename, np.ndarray)
    """
    pass


def draw_detections(image, boxes, color=(0, 255, 0), thickness=2):
    """
    Draw bounding boxes on a copy of image and return it.

    Parameters
    ----------
    image  : np.ndarray
    boxes  : np.ndarray, shape (N, 4) — [x1, y1, x2, y2]
    """
    pass


def evaluate(test_images, clf, scaler):
    """
    Run detection on all test images and print a summary of results.
    Visualise detections for the first few images.
    """
    pass


test_images = load_test_images()
evaluate(test_images, clf, scaler)